# Learning Tracker Agent with AgentCore Memory
This notebook demonstrates a Learning Tracker Agent that uses AWS AgentCore memory to:
- Track topics and learning progress
- Remember user learning preferences (videos vs documentation, pace)
- Pick up exactly where the learner left off in the next session
- Provide personalized recommendations based on learning history

## Section 1: Import Required Libraries

In [ ]:
from learning_tracker_agents import (
    add_topic,
    record_progress,
    get_learning_status,
    get_topic_recommendations,
    update_learning_preference,
    SYSTEM_PROMPT,
    MODEL_ID
)

print("✅ Learning tracker tools imported successfully")

In [ ]:
import logging
from boto3.session import Session
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

boto_session = Session()
REGION = boto_session.region_name

logger = logging.getLogger(__name__)

print(f"✅ AWS Region: {REGION}")

## Section 2: Memory Creation

This section creates AgentCore memory with two key strategies:
- **USER_PREFERENCE**: Stores learning pace and content format preferences
- **SEMANTIC**: Stores learned topics and progress information

In [ ]:
# Define memory name for identification
# WHY: so that we can uniquely identify and reuse this memory resource
memory_name = "LearningTrackerMemory"

# Initialize MemoryManager with region
# WHY: so that we can create/manage memory resources in the correct AWS region
memory_manager = MemoryManager(region_name=REGION)

# Create or fetch existing memory with defined strategies
# WHY: so that we don't create duplicate memory and can reuse existing one if already present
memory = memory_manager.get_or_create_memory(
    name=memory_name,
    strategies=[
        {
            # Define USER_PREFERENCE memory strategy
            # WHY: so that we can store and retrieve learner preferences for personalization
            StrategyType.USER_PREFERENCE.value: {
                "name": "LearnerPreferences",
                "description": "Captures learner preferences and learning style",
                # Define namespace for learner-specific preference storage
                # WHY: so that each learner's preferences are isolated using actorId
                "namespaces": ["learning/learner/{actorId}/preferences/"],
            }
        },
        {
            # Define SEMANTIC memory strategy
            # WHY: so that we can store factual information about topics and progress
            StrategyType.SEMANTIC.value: {
                "name": "LearningProgress",
                "description": "Stores topics learned, progress, and concepts covered",
                # Define namespace for learner-specific learning data
                # WHY: so that learning progress is stored per learner for future retrieval
                "namespaces": ["learning/learner/{actorId}/progress/"],
            }
        },
    ]
)

# Extract memory ID from response
# WHY: so that we can reference this memory in agent/runtime later
memory_id = memory["id"]

In [ ]:
if memory_id:
    print("✅ AgentCore Memory created successfully!")
    print(f"Memory ID: {memory_id}")
else:
    print("Memory resource not created. Try Again!")

In [ ]:
#!/usr/bin/python
"""AgentCore Memory integration for Learning Tracker agent."""

import uuid
from bedrock_agentcore.memory import MemoryClient
from strands.hooks import (
    AfterInvocationEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)

ACTOR_ID = "learner_001"
SESSION_ID = str(uuid.uuid4())

memory_client = MemoryClient(region_name=REGION)

class LearningTrackerMemoryHooks(HookProvider):
    """Memory hooks for learning tracker agent"""

    def __init__(
        self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str
    ):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_learning_context(self, event: MessageAddedEvent):
        """Retrieve learning context before processing user query"""
        messages = event.agent.messages
        if (
            messages[-1]["role"] == "user"
            and "toolResult" not in messages[-1]["content"][0]
        ):
            user_query = messages[-1]["content"][0]["text"]

            try:
                all_context = []

                for context_type, namespace in self.namespaces.items():
                    # *** AGENTCORE MEMORY USAGE *** - Retrieve learning context from each namespace
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,
                    )
                    # Post-processing: Format memories into context strings
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(
                                        f"[{context_type.upper()}] {text}"
                                    )

                # Inject learning context into the query
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = (
                        f"Learning Context:\n{context_text}\n\n{original_text}"
                    )
                    logger.info(f"Retrieved {len(all_context)} learning context items")

            except Exception as e:
                logger.error(f"Failed to retrieve learning context: {e}")

    def save_learning_interaction(self, event: AfterInvocationEvent):
        """Save learning interaction after agent response"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last user query and agent response
                learner_query = None
                agent_response = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif (
                        msg["role"] == "user"
                        and not learner_query
                        and "toolResult" not in msg["content"][0]
                    ):
                        learner_query = msg["content"][0]["text"]
                        break

                if learner_query and agent_response:
                    # *** AGENTCORE MEMORY USAGE *** - Save the learning interaction
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[
                            (learner_query, "USER"),
                            (agent_response, "ASSISTANT"),
                        ],
                    )
                    logger.info("Saved learning interaction to memory")

        except Exception as e:
            logger.error(f"Failed to save learning interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register learning tracker memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_learning_context)
        registry.add_callback(AfterInvocationEvent, self.save_learning_interaction)
        logger.info("Learning tracker memory hooks registered")

## Section 3: Seed Memory with Previous Learning Interactions
This section populates the memory with sample past interactions to simulate learning history

In [ ]:
# Seed with previous learner interactions (Learning Tracker Agent)
previous_interactions = [
    (
        "I'm learning Kubernetes. What have I covered so far?",
        "USER",
    ),
    (
        "Based on your learning journey in Kubernetes, you've covered Pods and Services. Let me help you track more progress.",
        "ASSISTANT",
    ),
    (
        "I've studied Pods and Services today. That took 3 hours.",
        "USER",
    ),
    (
        "Great! Progress recorded. You've covered Pods and Services in Kubernetes (3 hours). The next topics to cover are Deployments and StatefulSets.",
        "ASSISTANT",
    ),
    (
        "I prefer learning through videos. What's my learning pace?",
        "USER",
    ),
    (
        "Your preferences show you like video-based content. I'll prioritize video tutorials for Deployments and StatefulSets next.",
        "ASSISTANT",
    ),
    (
        "I completed Deployments yesterday. About 2 hours.",
        "USER",
    ),
    (
        "Excellent! Deployments completed (2 hours). You're making steady progress. Next is StatefulSets, which is crucial for Kubernetes.",
        "ASSISTANT",
    ),
    (
        "What topics should I learn next after Kubernetes?",
        "USER",
    ),
    (
        "Based on your DevOps learning path and video preference, I recommend Docker next as a foundation, followed by Terraform for Infrastructure as Code.",
        "ASSISTANT",
    ),
]

In [ ]:
# Check if memory_id exists before proceeding
# WHY: so that we don't attempt to write to memory without a valid memory resource
if memory_id:
    try:
        # Initialize MemoryClient
        # WHAT: low-level client to interact with Bedrock memory service
        # WHY: so that we can write events (conversations) into memory
        memory_client = MemoryClient(region_name=REGION)

        # Create a memory event (store conversation)
        # WHAT: sends past interactions to memory system
        # WHY: so that AWS can process and store them for future retrieval
        memory_client.create_event(
            memory_id=memory_id,  # Reference to the memory resource
            # WHY: tells AWS which memory system to store this data in

            actor_id=ACTOR_ID,
            # WHAT: unique learner identifier
            # WHY: so that memory is stored per learner (multi-tenant isolation)

            session_id="previous_session",
            # WHAT: identifier for this conversation session
            # WHY: helps group messages and track conversation context

            messages=previous_interactions,
            # WHAT: list of past user + assistant messages
            # WHY: so that AWS can extract preferences and progress from them
        )

        # Log success message
        # WHY: to confirm that seeding worked successfully
        print("✅ Seeded learning history successfully")

        # Inform that data first goes to Short-Term Memory
        # WHY: events are initially stored as session context before processing
        print("📝 Interactions saved to Short-Term Memory")

        # Inform about async Long-Term Memory processing
        # WHY: AWS will automatically extract preferences + semantic info in background
        print("⏳ Long-Term Memory processing will begin automatically...")

    except Exception as e:
        # Handle any errors during memory write
        # WHY: to avoid crash and help debugging
        print(f"⚠️ Error seeding history: {e}")

## Section 4: Check Long-Term Memory Processing
This section polls and verifies that long-term memory has processed the seeded interactions

In [ ]:
# Import time module
# WHY: so that we can wait between retries while checking async memory processing
import time

# Print initial check message
# WHY: to indicate we are starting to verify Long-Term Memory processing
print("🔍 Checking for processed Long-Term Memories...")

# Initialize retry counter
# WHY: to track how many times we have checked for memory availability
retries = 0

# Set maximum retries (6 × 10 seconds = 60 seconds)
# WHY: to avoid infinite waiting if memory processing is delayed
max_retries = 6  # 1 minute wait

# Start polling loop to check if LTM processing is complete
# WHY: because LTM processing is asynchronous and not immediately available
while retries < max_retries:

    # Retrieve memories from USER_PREFERENCE namespace
    # WHAT: querying extracted preference memories for this learner
    # WHY: to verify if AWS has processed STM → LTM
    memories = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"learning/learner/{ACTOR_ID}/preferences/",
        query="What are my learning preferences?",
    )

    # Check if any memories are returned
    # WHY: presence of data means LTM processing is complete
    if memories:
        # Print success message with count and time taken
        # WHY: to confirm memory extraction worked
        print(
            f"✅ Found {len(memories)} preference memories after {retries * 10} seconds!"
        )
        break

    # Increment retry count
    # WHY: to move to next polling attempt
    retries += 1

    # If retries still remain
    # WHY: continue waiting for async processing
    if retries < max_retries:
        # Inform user that processing is still ongoing
        print(
            f"⏳ Still processing... waiting 10 more seconds (attempt {retries}/{max_retries})"
        )
        # Wait for 10 seconds before next check
        # WHY: avoid excessive API calls and give time for processing
        time.sleep(10)
    else:
        # Max retries reached
        # WHY: handle delay or failure scenario gracefully
        print(
            "⚠️ Memory processing is taking longer than expected. Please try again in a moment."
        )
        break

# Print header for extracted preferences
# WHY: to display results in readable format
print(
    "\n🎯 AgentCore Memory automatically extracted these learner preferences from seeded conversations:"
)

# Print separator line
# WHY: improve readability of output
print("=" * 80)

# Loop through retrieved memories
# WHY: to display each extracted memory item
if memories:
    for i, memory in enumerate(memories, 1):

        # Ensure memory is in dictionary format
        # WHY: avoid runtime errors if structure differs
        if isinstance(memory, dict):

            # Extract content field
            # WHY: actual memory text is nested inside content
            content = memory.get("content", {})

            # Ensure content is dictionary
            # WHY: safe access to text field
            if isinstance(content, dict):

                # Extract text from memory
                # WHY: this is the actual extracted preference
                text = content.get("text", "")

                # Print memory item
                # WHY: display extracted preference to user
                print(f"  {i}. {text}")
else:
    print("  (Memories still processing or not yet available)")

## Section 5: Create Learning Tracker Agent
Initialize the agent with Bedrock model, memory configuration, tools, and system prompt

In [ ]:
import uuid
from strands import Agent
from strands.models import BedrockModel
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

# Generate a unique session ID
session_id = uuid.uuid4()

# Configure AgentCore memory
# WHAT: defines how agent will interact with memory (read + write)
# WHY: so that agent can retrieve past context and store new information automatically
memory_config = AgentCoreMemoryConfig(
    memory_id=memory_id,
    # WHAT: reference to memory resource created earlier
    # WHY: tells agent which memory system to use

    session_id=str(session_id),
    # WHAT: current session identifier
    # WHY: helps group conversation events in STM

    actor_id=ACTOR_ID,
    # WHAT: learner identifier
    # WHY: ensures memory is isolated per learner (multi-tenant)

    retrieval_config={
        # Configure semantic memory retrieval
        # WHAT: defines how many semantic memories to fetch and relevance threshold
        # WHY: so that only relevant progress/topic context is passed to LLM
        "learning/learner/{actorId}/progress/": RetrievalConfig(top_k=3, relevance_score=0.2),

        # Configure preference memory retrieval
        # WHAT: defines retrieval behavior for learner preferences
        # WHY: so that personalization context is included in LLM prompt
        "learning/learner/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.2)
    }
)

# Initialize Bedrock model
# WHAT: creates LLM instance using Claude or configured model
# WHY: so that agent can perform reasoning, tool selection, and response generation
model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

# Initialize memory hooks
memory_hooks = LearningTrackerMemoryHooks(
    memory_id=memory_id,
    client=memory_client,
    actor_id=ACTOR_ID,
    session_id=SESSION_ID,
)

# Create the learning tracker agent
# WHAT: combines model + memory + tools + system prompt
# WHY: so that we have a fully functional AI agent capable of reasoning, tool usage, and memory handling
agent = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[
        add_topic,
        record_progress,
        get_learning_status,
        get_topic_recommendations,
        update_learning_preference,
    ],
    hooks=[memory_hooks],
)

print("✅ Learning Tracker Agent is all set with LLM, Tools and Memory")

## Section 6: Test Learning Tracker Agent
Run test queries to demonstrate the agent's ability to recall progress, preferences, and continue learning sessions

In [ ]:
print("📚 Test 1: Check what's been covered in Kubernetes learning\n")
response1 = agent("I'm learning Kubernetes. What have I covered so far?")

In [ ]:
print(response1)

print("\n🎯 Test 2: Recall learning preferences\n")
response2 = agent("I prefer videos. Remember my learning pace and format.")
print(response2)

print("\n📈 Test 3: Ask for next learning step\n")
response3 = agent("What should I learn next after Kubernetes?")
print(response3)

print("\n📝 Test 4: Record a new progress update\n")
response4 = agent("I studied StatefulSets today and learned how services route traffic. It took 2 hours.")
print(response4)